In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

PROJECT_PATH = Path.cwd().parent
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"

print("Project path:", PROJECT_PATH)
print("Processed data path:", PROCESSED_PATH)

Project path: c:\Users\User\Desktop\foresight
Processed data path: c:\Users\User\Desktop\foresight\data\processed


In [2]:
sales = pd.read_csv(
    PROCESSED_PATH / "sales_daily_clean.csv",
    parse_dates=["Date"]
)

sales = sales.sort_values(
    ["SKU_Standard", "Date"]
).reset_index(drop=True)

print("Sales shape:", sales.shape)
print("Date range:", sales["Date"].min().date(), "to", sales["Date"].max().date())
print("SKUs:", sales["SKU_Standard"].nunique())

Sales shape: (36550, 7)
Date range: 2024-01-01 to 2025-12-31
SKUs: 50


In [3]:
weekly_sales = (
    sales.set_index("Date")
    .groupby("SKU_Standard")["Units_Sold"]
    .resample("W-SUN")
    .sum()
    .reset_index()
)

weekly_sales = weekly_sales.rename(
    columns={"Units_Sold": "Weekly_Units"}
)

weekly_sales = weekly_sales.sort_values(
    ["SKU_Standard", "Date"]
).reset_index(drop=True)

print("Weekly dataset shape:", weekly_sales.shape)
display(weekly_sales.head(10))

Weekly dataset shape: (5250, 3)


,SKU_Standard,Date,Weekly_Units
0,SKU00001,2024-01-07,106
1,SKU00001,2024-01-14,85
2,SKU00001,2024-01-21,117
3,SKU00001,2024-01-28,89
4,SKU00001,2024-02-04,119
5,SKU00001,2024-02-11,112
6,SKU00001,2024-02-18,104
7,SKU00001,2024-02-25,120
8,SKU00001,2024-03-03,103
9,SKU00001,2024-03-10,119


In [4]:
weekly_sales["Seasonal_Naive_Forecast"] = (
    weekly_sales
    .groupby("SKU_Standard")["Weekly_Units"]
    .shift(52)
)

print("Weekly dataset shape:", weekly_sales.shape)

display(
    weekly_sales[
        [
            "SKU_Standard",
            "Date",
            "Weekly_Units",
            "Seasonal_Naive_Forecast"
        ]
    ].tail(15)
)


Weekly dataset shape: (5250, 4)


,SKU_Standard,Date,Weekly_Units,Seasonal_Naive_Forecast
5235,SKU00050,2025-09-28,38,25.0
5236,SKU00050,2025-10-05,35,34.0
5237,SKU00050,2025-10-12,39,33.0
5238,SKU00050,2025-10-19,31,38.0
5239,SKU00050,2025-10-26,31,38.0
5240,SKU00050,2025-11-02,32,36.0
5241,SKU00050,2025-11-09,46,54.0
5242,SKU00050,2025-11-16,40,48.0
5243,SKU00050,2025-11-23,41,39.0
5244,SKU00050,2025-11-30,30,37.0


In [5]:
test_weeks = 13

all_weeks = sorted(
    weekly_sales["Date"].unique()
)

test_start_date = all_weeks[-test_weeks]

baseline_test = weekly_sales[
    (weekly_sales["Date"] >= test_start_date) &
    (weekly_sales["Seasonal_Naive_Forecast"].notna())
].copy()

print("Test start:", pd.Timestamp(test_start_date).date())
print("Test end  :", baseline_test["Date"].max().date())
print("Test rows :", len(baseline_test))
print("Test SKUs :", baseline_test["SKU_Standard"].nunique())

display(baseline_test.head(10))

Test start: 2025-10-12
Test end  : 2026-01-04
Test rows : 650
Test SKUs : 50


,SKU_Standard,Date,Weekly_Units,Seasonal_Naive_Forecast
92,SKU00001,2025-10-12,82,82.0
93,SKU00001,2025-10-19,82,88.0
94,SKU00001,2025-10-26,78,89.0
95,SKU00001,2025-11-02,93,83.0
96,SKU00001,2025-11-09,92,99.0
97,SKU00001,2025-11-16,87,80.0
98,SKU00001,2025-11-23,84,88.0
99,SKU00001,2025-11-30,84,68.0
100,SKU00001,2025-12-07,88,83.0
101,SKU00001,2025-12-14,89,94.0


In [6]:
def calculate_wape(actual, forecast):
    denominator = np.abs(actual).sum()

    if denominator == 0:
        return np.nan

    return (
        np.abs(actual - forecast).sum() /
        denominator
    ) * 100

In [7]:
baseline_wape = calculate_wape(
    baseline_test["Weekly_Units"],
    baseline_test["Seasonal_Naive_Forecast"]
)

baseline_mae = mean_absolute_error(
    baseline_test["Weekly_Units"],
    baseline_test["Seasonal_Naive_Forecast"]
)

print("SEASONAL-NAIVE BASELINE")
print("=" * 50)
print(f"Test period : {baseline_test['Date'].min().date()} to {baseline_test['Date'].max().date()}")
print(f"Test rows   : {len(baseline_test)}")
print(f"WAPE        : {baseline_wape:.2f}%")
print(f"MAE         : {baseline_mae:.2f} units")

SEASONAL-NAIVE BASELINE
Test period : 2025-10-12 to 2026-01-04
Test rows   : 650
WAPE        : 16.79%
MAE         : 13.50 units


In [8]:
# Exclude the final partial week because daily data ends on 2025-12-31

last_complete_week = pd.Timestamp("2025-12-28")

test_weeks = 13

complete_weeks = sorted(
    weekly_sales.loc[
        weekly_sales["Date"] <= last_complete_week,
        "Date"
    ].unique()
)

test_dates = complete_weeks[-test_weeks:]

baseline_test = weekly_sales[
    weekly_sales["Date"].isin(test_dates)
].copy()

baseline_test = baseline_test[
    baseline_test["Seasonal_Naive_Forecast"].notna()
].copy()

print("Test start:", baseline_test["Date"].min().date())
print("Test end  :", baseline_test["Date"].max().date())
print("Test rows :", len(baseline_test))
print("Test SKUs :", baseline_test["SKU_Standard"].nunique())

display(baseline_test.head(10))

Test start: 2025-10-05
Test end  : 2025-12-28
Test rows : 650
Test SKUs : 50


,SKU_Standard,Date,Weekly_Units,Seasonal_Naive_Forecast
91,SKU00001,2025-10-05,83,71.0
92,SKU00001,2025-10-12,82,82.0
93,SKU00001,2025-10-19,82,88.0
94,SKU00001,2025-10-26,78,89.0
95,SKU00001,2025-11-02,93,83.0
96,SKU00001,2025-11-09,92,99.0
97,SKU00001,2025-11-16,87,80.0
98,SKU00001,2025-11-23,84,88.0
99,SKU00001,2025-11-30,84,68.0
100,SKU00001,2025-12-07,88,83.0


In [9]:
baseline_wape = calculate_wape(
    baseline_test["Weekly_Units"],
    baseline_test["Seasonal_Naive_Forecast"]
)

baseline_mae = mean_absolute_error(
    baseline_test["Weekly_Units"],
    baseline_test["Seasonal_Naive_Forecast"]
)

print("SEASONAL-NAIVE BASELINE")
print("=" * 50)
print(
    f"Test period : "
    f"{baseline_test['Date'].min().date()} to "
    f"{baseline_test['Date'].max().date()}"
)
print(f"Test rows   : {len(baseline_test)}")
print(f"WAPE        : {baseline_wape:.2f}%")
print(f"MAE         : {baseline_mae:.2f} units")


SEASONAL-NAIVE BASELINE
Test period : 2025-10-05 to 2025-12-28
Test rows   : 650
WAPE        : 11.62%
MAE         : 9.73 units


In [10]:
model_data = weekly_sales.copy()

grouped_demand = model_data.groupby("SKU_Standard")["Weekly_Units"]

model_data["Lag_1"] = grouped_demand.shift(1)
model_data["Lag_2"] = grouped_demand.shift(2)
model_data["Lag_4"] = grouped_demand.shift(4)
model_data["Lag_8"] = grouped_demand.shift(8)
model_data["Lag_13"] = grouped_demand.shift(13)
model_data["Lag_52"] = grouped_demand.shift(52)

print("Lag features created.")
display(model_data.head(15))

Lag features created.


,SKU_Standard,Date,Weekly_Units,Seasonal_Naive_Forecast,Lag_1,Lag_2,Lag_4,Lag_8,Lag_13,Lag_52
0,SKU00001,2024-01-07,106,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SKU00001,2024-01-14,85,NaN,106.0,NaN,NaN,NaN,NaN,NaN
2,SKU00001,2024-01-21,117,NaN,85.0,106.0,NaN,NaN,NaN,NaN
3,SKU00001,2024-01-28,89,NaN,117.0,85.0,NaN,NaN,NaN,NaN
4,SKU00001,2024-02-04,119,NaN,89.0,117.0,106.0,NaN,NaN,NaN
5,SKU00001,2024-02-11,112,NaN,119.0,89.0,85.0,NaN,NaN,NaN
6,SKU00001,2024-02-18,104,NaN,112.0,119.0,117.0,NaN,NaN,NaN
7,SKU00001,2024-02-25,120,NaN,104.0,112.0,89.0,NaN,NaN,NaN
8,SKU00001,2024-03-03,103,NaN,120.0,104.0,119.0,106.0,NaN,NaN
9,SKU00001,2024-03-10,119,NaN,103.0,120.0,112.0,85.0,NaN,NaN


In [11]:
model_data["Year"] = model_data["Date"].dt.year
model_data["Month"] = model_data["Date"].dt.month
model_data["Quarter"] = model_data["Date"].dt.quarter
model_data["WeekOfYear"] = model_data["Date"].dt.isocalendar().week.astype(int)

print("Calendar features created.")
display(
    model_data[
        [
            "SKU_Standard",
            "Date",
            "Year",
            "Month",
            "Quarter",
            "WeekOfYear"
        ]
    ].head()
)

Calendar features created.


,SKU_Standard,Date,Year,Month,Quarter,WeekOfYear
0,SKU00001,2024-01-07,2024,1,1,1
1,SKU00001,2024-01-14,2024,1,1,2
2,SKU00001,2024-01-21,2024,1,1,3
3,SKU00001,2024-01-28,2024,1,1,4
4,SKU00001,2024-02-04,2024,2,1,5


In [12]:
weekly_promo = (
    sales.assign(
        Week=sales["Date"].dt.to_period("W-SUN").dt.end_time.dt.normalize()
    )
    .groupby(["SKU_Standard", "Week"])["Promotion"]
    .max()
    .reset_index()
    .rename(
        columns={
            "Week": "Date",
            "Promotion": "Promotion_Active"
        }
    )
)

model_data = model_data.merge(
    weekly_promo,
    on=["SKU_Standard", "Date"],
    how="left"
)

model_data["Promotion_Active"] = (
    model_data["Promotion_Active"]
    .fillna(0)
    .astype(int)
)

print("Promotion feature added.")
display(
    model_data[
        [
            "SKU_Standard",
            "Date",
            "Weekly_Units",
            "Promotion_Active"
        ]
    ].head(10)
)

Promotion feature added.


,SKU_Standard,Date,Weekly_Units,Promotion_Active
0,SKU00001,2024-01-07,106,0
1,SKU00001,2024-01-14,85,0
2,SKU00001,2024-01-21,117,0
3,SKU00001,2024-01-28,89,0
4,SKU00001,2024-02-04,119,0
5,SKU00001,2024-02-11,112,0
6,SKU00001,2024-02-18,104,0
7,SKU00001,2024-02-25,120,0
8,SKU00001,2024-03-03,103,1
9,SKU00001,2024-03-10,119,1


In [13]:
print("MODEL FEATURE DATASET")
print("=" * 50)

print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))
print("Missing values:")

display(
    model_data.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

MODEL FEATURE DATASET
Rows: 5250
Columns: 15
Missing values:


Seasonal_Naive_Forecast    2600
Lag_52                     2600
Lag_13                      650
Lag_8                       400
Lag_4                       200
Lag_2                       100
Lag_1                        50
Date                          0
SKU_Standard                  0
Weekly_Units                  0
Year                          0
Month                         0
Quarter                       0
WeekOfYear                    0
Promotion_Active              0
dtype: int64

In [15]:
# Create a fresh copy
model_data = weekly_sales.copy()

# Grouped demand series
grouped_demand = model_data.groupby("SKU_Standard")["Weekly_Units"]

# -----------------------------
# Lag Features
# -----------------------------

model_data["Lag_1"] = grouped_demand.shift(1)
model_data["Lag_2"] = grouped_demand.shift(2)
model_data["Lag_4"] = grouped_demand.shift(4)
model_data["Lag_8"] = grouped_demand.shift(8)
model_data["Lag_13"] = grouped_demand.shift(13)
model_data["Lag_52"] = grouped_demand.shift(52)

# -----------------------------
# Rolling Features
# -----------------------------

model_data["Rolling_Mean_4"] = (
    model_data.groupby("SKU_Standard")["Weekly_Units"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

model_data["Rolling_Mean_8"] = (
    model_data.groupby("SKU_Standard")["Weekly_Units"]
    .transform(lambda x: x.shift(1).rolling(8).mean())
)

model_data["Rolling_Std_8"] = (
    model_data.groupby("SKU_Standard")["Weekly_Units"]
    .transform(lambda x: x.shift(1).rolling(8).std())
)

# -----------------------------
# Calendar Features
# -----------------------------

model_data["Year"] = model_data["Date"].dt.year
model_data["Month"] = model_data["Date"].dt.month
model_data["Quarter"] = model_data["Date"].dt.quarter
model_data["WeekOfYear"] = (
    model_data["Date"].dt.isocalendar().week.astype(int)
)

# -----------------------------
# Promotion Feature
# -----------------------------

weekly_promo = (
    sales.assign(
        Week=sales["Date"]
        .dt.to_period("W-SUN")
        .dt.end_time
        .dt.normalize()
    )
    .groupby(["SKU_Standard", "Week"])["Promotion"]
    .max()
    .reset_index()
    .rename(
        columns={
            "Week": "Date",
            "Promotion": "Promotion_Active"
        }
    )
)

model_data = model_data.merge(
    weekly_promo,
    on=["SKU_Standard", "Date"],
    how="left"
)

model_data["Promotion_Active"] = (
    model_data["Promotion_Active"]
    .fillna(0)
    .astype(int)
)

# Sort again
model_data = model_data.sort_values(
    ["SKU_Standard", "Date"]
).reset_index(drop=True)

print("Feature engineering completed.")
print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))
print("\nColumns:")
print(model_data.columns.tolist())

Feature engineering completed.
Rows: 5250
Columns: 18

Columns:
['SKU_Standard', 'Date', 'Weekly_Units', 'Seasonal_Naive_Forecast', 'Lag_1', 'Lag_2', 'Lag_4', 'Lag_8', 'Lag_13', 'Lag_52', 'Rolling_Mean_4', 'Rolling_Mean_8', 'Rolling_Std_8', 'Year', 'Month', 'Quarter', 'WeekOfYear', 'Promotion_Active']


In [16]:
print(
    model_data[
        [
            "SKU_Standard",
            "Date",
            "Weekly_Units",
            "Lag_1",
            "Lag_52",
            "Rolling_Mean_4",
            "Rolling_Mean_8",
            "Rolling_Std_8"
        ]
    ].head(15)
)

   SKU_Standard       Date  Weekly_Units  Lag_1  Lag_52  Rolling_Mean_4  \
0      SKU00001 2024-01-07           106    NaN     NaN             NaN   
1      SKU00001 2024-01-14            85  106.0     NaN             NaN   
2      SKU00001 2024-01-21           117   85.0     NaN             NaN   
3      SKU00001 2024-01-28            89  117.0     NaN             NaN   
4      SKU00001 2024-02-04           119   89.0     NaN           99.25   
5      SKU00001 2024-02-11           112  119.0     NaN          102.50   
6      SKU00001 2024-02-18           104  112.0     NaN          109.25   
7      SKU00001 2024-02-25           120  104.0     NaN          106.00   
8      SKU00001 2024-03-03           103  120.0     NaN          113.75   
9      SKU00001 2024-03-10           119  103.0     NaN          109.75   
10     SKU00001 2024-03-17           122  119.0     NaN          111.50   
11     SKU00001 2024-03-24           130  122.0     NaN          116.00   
12     SKU00001 2024-03-3

In [17]:
test_dates = sorted(
    baseline_test["Date"].unique()
)

model_data_clean = model_data.dropna(
    subset=[
        "Lag_1",
        "Lag_2",
        "Lag_4",
        "Lag_8",
        "Lag_13",
        "Lag_52",
        "Rolling_Mean_4",
        "Rolling_Mean_8",
        "Rolling_Std_8"
    ]
).copy()

train_data = model_data_clean[
    ~model_data_clean["Date"].isin(test_dates)
].copy()

test_data = model_data_clean[
    model_data_clean["Date"].isin(test_dates)
].copy()

print("TRAINING DATA")
print("Rows:", len(train_data))

print("\nTEST DATA")
print("Rows:", len(test_data))

print("\nTraining date range:")
print(train_data["Date"].min(), "to", train_data["Date"].max())

print("\nTesting date range:")
print(test_data["Date"].min(), "to", test_data["Date"].max())

TRAINING DATA
Rows: 2000

TEST DATA
Rows: 650

Training date range:
2025-01-05 00:00:00 to 2026-01-04 00:00:00

Testing date range:
2025-10-05 00:00:00 to 2025-12-28 00:00:00


In [18]:
# Remove the incomplete week ending 2026-01-04
model_data_clean = model_data_clean[
    model_data_clean["Date"] <= pd.Timestamp("2025-12-28")
].copy()

# Recreate train and test sets
train_data = model_data_clean[
    ~model_data_clean["Date"].isin(test_dates)
].copy()

test_data = model_data_clean[
    model_data_clean["Date"].isin(test_dates)
].copy()

print("TRAINING DATA")
print("Rows:", len(train_data))
print(
    "Date range:",
    train_data["Date"].min(),
    "to",
    train_data["Date"].max()
)

print("\nTEST DATA")
print("Rows:", len(test_data))
print(
    "Date range:",
    test_data["Date"].min(),
    "to",
    test_data["Date"].max()
)

TRAINING DATA
Rows: 1950
Date range: 2025-01-05 00:00:00 to 2025-09-28 00:00:00

TEST DATA
Rows: 650
Date range: 2025-10-05 00:00:00 to 2025-12-28 00:00:00


In [19]:
feature_columns = [
    "Lag_1",
    "Lag_2",
    "Lag_4",
    "Lag_8",
    "Lag_13",
    "Lag_52",
    "Rolling_Mean_4",
    "Rolling_Mean_8",
    "Rolling_Std_8",
    "Year",
    "Month",
    "Quarter",
    "WeekOfYear",
    "Promotion_Active"
]

X_train = train_data[feature_columns]
y_train = train_data["Weekly_Units"]

X_test = test_data[feature_columns]
y_test = test_data["Weekly_Units"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


X_train: (1950, 14)
y_train: (1950,)
X_test: (650, 14)
y_test: (650,)


In [20]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed successfully.")

Model training completed successfully.


In [21]:
test_data = test_data.copy()

test_data["ML_Forecast"] = model.predict(X_test)

# Demand cannot be negative
test_data["ML_Forecast"] = test_data["ML_Forecast"].clip(lower=0)

display(
    test_data[
        [
            "SKU_Standard",
            "Date",
            "Weekly_Units",
            "ML_Forecast"
        ]
    ].head(15)
)

,SKU_Standard,Date,Weekly_Units,ML_Forecast
91,SKU00001,2025-10-05,83,76.049474
92,SKU00001,2025-10-12,82,79.120468
93,SKU00001,2025-10-19,82,74.481890
94,SKU00001,2025-10-26,78,75.866725
95,SKU00001,2025-11-02,93,78.517119
96,SKU00001,2025-11-09,92,81.156549
97,SKU00001,2025-11-16,87,80.003520
98,SKU00001,2025-11-23,84,80.314656
99,SKU00001,2025-11-30,84,81.447840
100,SKU00001,2025-12-07,88,75.714642


In [22]:
from sklearn.metrics import mean_absolute_error

ml_wape = calculate_wape(
    test_data["Weekly_Units"],
    test_data["ML_Forecast"]
)

ml_mae = mean_absolute_error(
    test_data["Weekly_Units"],
    test_data["ML_Forecast"]
)

improvement = (
    (baseline_wape - ml_wape)
    / baseline_wape
) * 100

print("FORECAST MODEL RESULTS")
print("----------------------")
print(f"Seasonal Naive WAPE : {baseline_wape:.2f}%")
print(f"ML Model WAPE       : {ml_wape:.2f}%")
print(f"Seasonal Naive MAE  : {baseline_test['Weekly_Units'].sub(baseline_test['Seasonal_Naive_Forecast']).abs().mean():.2f}")
print(f"ML Model MAE        : {ml_mae:.2f}")
print(f"Improvement         : {improvement:.2f}%")

FORECAST MODEL RESULTS
----------------------
Seasonal Naive WAPE : 11.62%
ML Model WAPE       : 10.29%
Seasonal Naive MAE  : 9.73
ML Model MAE        : 8.62
Improvement         : 11.44%


In [23]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Use several historical forecast cutoffs
validation_periods = [
    pd.Timestamp("2025-06-29"),
    pd.Timestamp("2025-07-27"),
    pd.Timestamp("2025-08-31")
]

rolling_results = []

for cutoff in validation_periods:

    # Training data must be strictly before the validation period
    fold_train = model_data_clean[
        model_data_clean["Date"] < cutoff
    ].copy()

    # Validate on the next 4 weeks
    validation_end = cutoff + pd.Timedelta(weeks=4)

    fold_test = model_data_clean[
        (model_data_clean["Date"] >= cutoff) &
        (model_data_clean["Date"] < validation_end)
    ].copy()

    if len(fold_test) == 0:
        continue

    X_fold_train = fold_train[feature_columns]
    y_fold_train = fold_train["Weekly_Units"]

    X_fold_test = fold_test[feature_columns]
    y_fold_test = fold_test["Weekly_Units"]

    fold_model = HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    fold_model.fit(X_fold_train, y_fold_train)

    fold_test["Forecast"] = (
        fold_model.predict(X_fold_test)
        .clip(min=0)
    )

    fold_wape = calculate_wape(
        fold_test["Weekly_Units"],
        fold_test["Forecast"]
    )

    rolling_results.append({
        "Cutoff": cutoff,
        "Validation_Start": fold_test["Date"].min(),
        "Validation_End": fold_test["Date"].max(),
        "Rows": len(fold_test),
        "WAPE": fold_wape
    })

rolling_results_df = pd.DataFrame(rolling_results)

display(rolling_results_df)

print(
    "\nAverage Rolling-Origin WAPE:",
    f"{rolling_results_df['WAPE'].mean():.2f}%"
)

,Cutoff,Validation_Start,Validation_End,Rows,WAPE
0,2025-06-29,2025-06-29,2025-07-20,200,10.924319
1,2025-07-27,2025-07-27,2025-08-17,200,10.085396
2,2025-08-31,2025-08-31,2025-09-21,200,10.475870



Average Rolling-Origin WAPE: 10.50%


In [24]:
# Final model training on all available historical data
final_train = model_data_clean.copy()

X_final = final_train[feature_columns]
y_final = final_train["Weekly_Units"]

final_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

final_model.fit(X_final, y_final)

print("Final forecasting model trained.")
print("Training rows:", len(final_train))

Final forecasting model trained.
Training rows: 2600


In [25]:
# Generate next 4 weekly SKU forecasts

future_weeks = pd.date_range(
    start="2026-01-04",
    periods=4,
    freq="W-SUN"
)

future_rows = []

# Keep historical demand for recursive lag calculation
history = weekly_sales[
    ["SKU_Standard", "Date", "Weekly_Units"]
].copy()

for sku in history["SKU_Standard"].unique():

    sku_history = history[
        history["SKU_Standard"] == sku
    ].sort_values("Date").copy()

    demand_history = list(sku_history["Weekly_Units"].values)

    for future_date in future_weeks:

        # Calculate features from historical/previous predictions
        lag_1 = demand_history[-1]
        lag_2 = demand_history[-2]
        lag_4 = demand_history[-4]
        lag_8 = demand_history[-8]
        lag_13 = demand_history[-13]
        lag_52 = demand_history[-52]

        rolling_mean_4 = np.mean(demand_history[-4:])
        rolling_mean_8 = np.mean(demand_history[-8:])
        rolling_std_8 = np.std(
            demand_history[-8:],
            ddof=1
        )

        # Future promotion information is not available,
        # so use 0 as the default.
        promotion_active = 0

        X_future = pd.DataFrame([{
            "Lag_1": lag_1,
            "Lag_2": lag_2,
            "Lag_4": lag_4,
            "Lag_8": lag_8,
            "Lag_13": lag_13,
            "Lag_52": lag_52,
            "Rolling_Mean_4": rolling_mean_4,
            "Rolling_Mean_8": rolling_mean_8,
            "Rolling_Std_8": rolling_std_8,
            "Year": future_date.year,
            "Month": future_date.month,
            "Quarter": future_date.quarter,
            "WeekOfYear": int(future_date.isocalendar().week),
            "Promotion_Active": promotion_active
        }])

        forecast = final_model.predict(X_future)[0]
        forecast = max(0, forecast)

        future_rows.append({
            "SKU_Standard": sku,
            "Date": future_date,
            "Forecast_Units": forecast
        })

        # Add prediction to history for recursive forecasting
        demand_history.append(forecast)

future_forecast = pd.DataFrame(future_rows)

print("Future forecast generated successfully.")
print("Rows:", len(future_forecast))

display(future_forecast.head(20))


Future forecast generated successfully.
Rows: 200


,SKU_Standard,Date,Forecast_Units
0,SKU00001,2026-01-04,79.150757
1,SKU00001,2026-01-11,75.486975
2,SKU00001,2026-01-18,75.968063
3,SKU00001,2026-01-25,77.210258
4,SKU00002,2026-01-04,56.799727
5,SKU00002,2026-01-11,64.752218
6,SKU00002,2026-01-18,60.246601
7,SKU00002,2026-01-25,66.673171
8,SKU00003,2026-01-04,33.395832
9,SKU00003,2026-01-11,33.229577


In [26]:
future_forecast.to_csv(
    PROCESSED_PATH / "future_weekly_forecast.csv",
    index=False
)

print("Saved:")
print(PROCESSED_PATH / "future_weekly_forecast.csv")

Saved:
c:\Users\User\Desktop\foresight\data\processed\future_weekly_forecast.csv


In [27]:
# ==========================================
# D4 — Inventory Risk Scoring
# ==========================================

future_forecast = pd.read_csv(
    PROCESSED_PATH / "future_weekly_forecast.csv",
    parse_dates=["Date"]
)

inventory = pd.read_csv(
    PROCESSED_PATH / "inventory_snapshots_clean.csv",
    parse_dates=["Snapshot_Date"]
)

print("Future forecast:", future_forecast.shape)
print("Inventory:", inventory.shape)

display(inventory.head())

Future forecast: (200, 3)
Inventory: (4800, 9)


,Snapshot_Date,SKU,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,Reorder_Point,Inventory_Value,SKU_Standard
0,2024-01-01,SKU001,16,23,11,7,18,58420.96,SKU00001
1,2024-01-01,SKU002,29,12,4,5,10,163983.40,SKU00002
2,2024-01-01,SKU003,34,23,14,6,20,209060.22,SKU00003
3,2024-01-01,SKU004,34,4,7,5,12,42257.92,SKU00004
4,2024-01-01,SKU005,23,5,5,6,11,170945.89,SKU00005


In [30]:
# ==========================================
# FORESIGHT D4 - INVENTORY RISK SCORING
# ==========================================

import numpy as np
import pandas as pd

# 1. Latest inventory snapshot for each SKU
latest_inventory = (
    inventory
    .sort_values("Snapshot_Date")
    .groupby("SKU_Standard")
    .tail(1)
    .copy()
)

# 2. Four-week forecast by SKU
four_week_demand = (
    future_forecast
    .groupby("SKU_Standard")["Forecast_Units"]
    .sum()
    .reset_index()
    .rename(columns={
        "Forecast_Units": "Forecast_4W_Units"
    })
)

# 3. Keep only SKUs that have both inventory and forecast
risk_data = latest_inventory.merge(
    four_week_demand,
    on="SKU_Standard",
    how="inner"
)

# 4. Calculate weekly demand
risk_data["Forecast_Weekly_Units"] = (
    risk_data["Forecast_4W_Units"] / 4
)

# 5. Estimate demand during lead time
risk_data["Lead_Time_Demand"] = (
    risk_data["Forecast_Weekly_Units"]
    * risk_data["Lead_Time_Days"]
    / 7
)

# 6. Available stock = current stock + incoming stock
risk_data["Available_Stock"] = (
    risk_data["Current_Stock"]
    + risk_data["On_Order"]
)

# 7. Stock coverage in weeks
risk_data["Stock_Coverage_Weeks"] = (
    risk_data["Available_Stock"]
    / risk_data["Forecast_Weekly_Units"].replace(0, np.nan)
)

# 8. Stockout risk
risk_data["Stockout_Risk"] = (
    risk_data["Available_Stock"]
    <
    (
        risk_data["Lead_Time_Demand"]
        + risk_data["Safety_Stock"]
    )
)

# 9. Overstock risk
risk_data["Overstock_Risk"] = (
    risk_data["Current_Stock"]
    >
    (
        risk_data["Forecast_4W_Units"] * 1.5
    )
)

# 10. Recommended action
def determine_action(row):
    if row["Stockout_Risk"]:
        return "Reorder Now"
    elif row["Overstock_Risk"]:
        return "Markdown / Clear"
    elif row["Stock_Coverage_Weeks"] < 4:
        return "Watch / Volatile"
    else:
        return "Healthy"

risk_data["Recommended_Action"] = (
    risk_data.apply(determine_action, axis=1)
)

# 11. Save D4 output
risk_data.to_csv(
    PROCESSED_PATH / "inventory_risk_scores.csv",
    index=False
)

print("D4 risk scoring completed successfully!")
print("Risk records:", len(risk_data))
print("Saved to:", PROCESSED_PATH / "inventory_risk_scores.csv")

D4 risk scoring completed successfully!
Risk records: 50
Saved to: c:\Users\User\Desktop\foresight\data\processed\inventory_risk_scores.csv


In [31]:
print("===== FORESIGHT D4 RISK SUMMARY =====")

print("Total SKUs:", len(risk_data))

print("\nStockout Risk:")
print(risk_data["Stockout_Risk"].value_counts())

print("\nOverstock Risk:")
print(risk_data["Overstock_Risk"].value_counts())

print("\nRecommended Actions:")
print(risk_data["Recommended_Action"].value_counts())

print(
    "\nAverage 4-Week Forecast:",
    round(risk_data["Forecast_4W_Units"].mean(), 2)
)

print(
    "\nTotal 4-Week Forecast:",
    round(risk_data["Forecast_4W_Units"].sum(), 2)
)

===== FORESIGHT D4 RISK SUMMARY =====
Total SKUs: 50

Stockout Risk:
Stockout_Risk
False    45
True      5
Name: count, dtype: int64

Overstock Risk:
Overstock_Risk
False    38
True     12
Name: count, dtype: int64

Recommended Actions:
Recommended_Action
Healthy             20
Watch / Volatile    13
Markdown / Clear    12
Reorder Now          5
Name: count, dtype: int64

Average 4-Week Forecast: 332.2

Total 4-Week Forecast: 16610.21


In [32]:
print(future_forecast.columns.tolist())
print(future_forecast.head())

['SKU_Standard', 'Date', 'Forecast_Units']
  SKU_Standard       Date  Forecast_Units
0     SKU00001 2026-01-04       79.150757
1     SKU00001 2026-01-11       75.486975
2     SKU00001 2026-01-18       75.968063
3     SKU00001 2026-01-25       77.210258
4     SKU00002 2026-01-04       56.799727


In [33]:
print(pd.read_csv(
    PROCESSED_PATH / "future_weekly_forecast.csv"
).columns.tolist())

['SKU_Standard', 'Date', 'Forecast_Units']
